# Continual-learning reversal task, run at the retention-task PICKS operating point

10-reversal continual-learning task (`ssb.envs.continual.run_cl`), with each rule's
`(alpha, beta, eta, tau)` taken from `f3_continual.ipynb`'s `PICKS` (its final summary figure) —
not from `F3_continual 2.ipynb`'s own CL-tuned values. Order: BCM, OJA, LC, PC, COV, HZ, HR.

In [ ]:
# ---- imports ----
from pathlib import Path
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from ssb.utils import batch_inits
from ssb.models import rules
from ssb.envs import continual as ctn

In [ ]:
# =====================================================================
#  CONFIG
# =====================================================================
FIGDIR = Path('.')
SAVE   = True
CDIR   = Path('cl_data_picks'); CDIR.mkdir(exist_ok=True)

ORDER = ['BCM', 'OJA', 'LC', 'PC', 'COV', 'HZ', 'HR']   # plotting order (requested)
LABEL = {'PC': 'PC', 'HR': 'RPE', 'COV': 'COV', 'OJA': 'OJA', 'BCM': 'BCM', 'LC': 'LC', 'HZ': 'LPL'}

# Verbatim from f3_continual.ipynb's cache/final_{rule}.npz -- the (eta, alpha, beta, tau) each
# rule's PICKS cell resolved to for that notebook's final summary figure (section 4-7). BCM/OJA/LC
# have no real det-sweep plateau there, so their PICKS fall back to best_params on the SINGLE-
# CONTEXT forgetting task -- see the eta caveat in the markdown above.
PICKS = {
    'BCM': dict(eta=1e-05,  alpha=0.0372759,  beta=1.0,        tau=0.001),
    'OJA': dict(eta=1e-05,  alpha=1.0,        beta=1.0,        tau=0.0),
    'LC':  dict(eta=0.0001, alpha=0.00061054, beta=1.0,        tau=0.00464159),
    'PC':  dict(eta=0.01,   alpha=0.0372759,  beta=0.00719686, tau=0.0),
    'COV': dict(eta=0.01,   alpha=0.0163789,  beta=0.0848343,  tau=0.1),
    'HZ':  dict(eta=0.01,   alpha=0.4394,     beta=0.007197,   tau=0.004642),
    'HR':  dict(eta=0.01,   alpha=0.439397,   beta=1.0,        tau=0.0),
}

SEEDS  = 100     # paper scale, matching F3_continual 2.ipynb
TS     = 50000   # trials per reversal block (bumped from 20000 -- check whether slower rules,
                 # e.g. RPE, just need more time per block to settle)
NR     = 10      # number of reversals
SEED   = 233
STRIDE = 40      # thin stored traces by this factor
MCLIP  = 0.3     # memory / relearn saturate at |dw|=MCLIP

CH_A, CH_B = '#8B6BA8', '#2a7fb8'   # channel-A / channel-B trace colors (identity, not performance)

# Per-rule identity color (fixed by rule name, independent of ORDER or outcome) -- used
# consistently across the bar charts and the scatter so a rule is always the same color.
CAT_COLOR = {
    'BCM': '#4C72B0', 'OJA': '#DD8452', 'LC': '#55A868', 'PC': '#C44E52',
    'COV': '#8172B2', 'HZ': '#937860', 'HR': '#DA8BC3',
}

plt.rcParams.update({'font.family': 'DejaVu Sans', 'mathtext.fontset': 'stixsans', 'svg.fonttype': 'none',
                     'font.size': 7, 'axes.linewidth': 0.25, 'xtick.major.width': 0.25,
                     'ytick.major.width': 0.25, 'xtick.major.size': 2, 'ytick.major.size': 2})

def nature_ax(ax, xlabel='', ylabel='', nx=4, ny=4):
    for sp in ('top', 'right'): ax.spines[sp].set_visible(False)
    for sp in ('left', 'bottom'): ax.spines[sp].set_linewidth(0.25)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(nx)); ax.yaxis.set_major_locator(mticker.MaxNLocator(ny))
    ax.tick_params(labelsize=7, width=0.25)
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)

In [ ]:
def run(rule):
    p = PICKS[rule]
    hps = batch_inits.make_default_hps(SEED)
    hps['sizes'] = ctn.CL_SIZES
    hps = rules.set_pars(hps, p['alpha'], p['beta'], p['eta'])
    hps['tau'] = jnp.ones_like(hps['tau']) * p['tau']
    hb = batch_inits.batchify_hps(hps, SEEDS)
    logs = ctn.run_cl(hb, rule, TS, separate_inputs=True, num_revs=NR)

    d0 = np.zeros((SEEDS, NR, TS)); d1 = np.zeros((SEEDS, NR, TS))
    end_W = np.zeros((SEEDS, NR, 2, 2)); acc = np.zeros((SEEDS, NR))
    for r in range(NR):
        a = np.asarray(logs[r][0]).T
        w = np.swapaxes(np.asarray(logs[r][1]), 0, 1)          # (S,TS,2,2)
        d0[:, r] = w[:, :, 0, 1] - w[:, :, 0, 0]                # channel-A pref (want <0)
        d1[:, r] = w[:, :, 1, 1] - w[:, :, 1, 0]                # channel-B pref (want >0)
        end_W[:, r] = w[:, -1]; acc[:, r] = (a[:, TS // 2:] == (0 if r % 2 == 0 else 1)).mean(1)
    D0 = d0.reshape(SEEDS, NR * TS)[:, ::STRIDE]; D1 = d1.reshape(SEEDS, NR * TS)[:, ::STRIDE]
    band = lambda X: (np.median(X, 0), np.percentile(X, 35, 0), np.percentile(X, 65, 0))
    m0, lo0, hi0 = band(D0); m1, lo1, hi1 = band(D1)
    return dict(m0=m0, lo0=lo0, hi0=hi0, m1=m1, lo1=lo1, hi1=hi1, end_W=end_W, acc=acc,
                Tp=D0.shape[1] // NR)


def _cache_path(rule):
    return CDIR / f"picks_{rule}_ts{TS}_seeds{SEEDS}.npz"


_CACHE_KEYS = ['m0', 'lo0', 'hi0', 'm1', 'lo1', 'hi1', 'end_W', 'acc', 'Tp']

def _load_or_run(rule):
    p = _cache_path(rule)
    if p.exists():
        z = np.load(p, allow_pickle=True)
        if all(k in z for k in _CACHE_KEYS):
            return {k: z[k] for k in _CACHE_KEYS}
    d = run(rule)
    np.savez(p, S=SEEDS, TS=TS, NR=NR, **d)
    return d


data = {}
for rule in ORDER:
    data[rule] = _load_or_run(rule)
    print(f"{LABEL[rule]:4s}  acc={data[rule]['acc'].mean():.2f}")

## Continual-learning reversal traces
Channel-A (purple) / channel-B (blue) preference $\Delta w$, first 5 of 10 reversal blocks shown.

In [ ]:
N_SHOW = 5       # display only the first 5 of the NR=10 simulated reversal blocks (clarity only)
PLOT_STRIDE = 8  # further thin the already-cached traces for plotting only (lighter/smaller SVG,
                 # less jagged line) -- doesn't touch the underlying data or its STRIDE=40 binning

def trace_panel(ax, rule, ylab=False):
    z = data[rule]; Tp = int(z['Tp'])
    n_pts = N_SHOW * Tp
    t = np.arange(n_pts)[::PLOT_STRIDE]
    ss = slice(2 * Tp, None)   # y-scale from the full run's settled region (blocks 2-10)
    lim = max(np.percentile(np.abs(np.concatenate([z['m0'][ss], z['m1'][ss]])), 95) * 1.4, 0.02)
    # Clip the DATA itself to the axis range before plotting. The early-block transient (block 0,
    # excluded from the lim calc above) can be several times larger than `lim` for some rules
    # (e.g. LPL, RPE) -- rather than relying on the renderer to clip that overflow at draw time
    # (which left stray line segments trailing off past the axes frame in the saved SVG, visible
    # in Illustrator as content clipped at the artboard edge), clip the values themselves so
    # nothing plotted ever exceeds [-lim, lim].
    clip = lambda a: np.clip(a[:n_pts:PLOT_STRIDE], -lim, lim)
    for k in range(1, N_SHOW): ax.axvline(k * Tp, color='#eee', lw=0.4, zorder=0)
    ax.axhline(0, color='#bbb', lw=0.4)
    ax.fill_between(t, clip(z['lo0']), clip(z['hi0']), color=CH_A, alpha=.12, lw=0)
    ax.fill_between(t, clip(z['lo1']), clip(z['hi1']), color=CH_B, alpha=.12, lw=0)
    ax.plot(t, clip(z['m0']), color=CH_A, lw=1.0, label='context A')
    ax.plot(t, clip(z['m1']), color=CH_B, lw=1.0, label='context B')
    ax.set_ylim(-lim, lim); ax.set_xlim(0, n_pts); ax.set_xticklabels([])
    nature_ax(ax, 'reversals →', 'channel pref. $\\Delta w$' if ylab else '', nx=3, ny=4)
    ax.set_title(LABEL[rule], fontsize=9, color='#222')

fig = plt.figure(figsize=(2.5 * len(ORDER), 2.1))
gs = fig.add_gridspec(1, len(ORDER), wspace=0.55)
for i, r in enumerate(ORDER):
    trace_panel(fig.add_subplot(gs[0, i]), r, ylab=(i == 0))
fig.axes[0].legend(fontsize=7, frameon=False, loc='upper left', handlelength=1.0, labelspacing=0.2, borderaxespad=0.15)
if SAVE:
    fig.savefig(FIGDIR / 'F3_picks_traces.svg', bbox_inches='tight')
plt.show()

## Held memory during the opposite block vs. behavioral performance
$y$ = each channel's own-signed weight preference while idle (weaker of the two), **as a fraction
of that rule's own achievable scale** (ratio undefined when a rule's own scale is noise-floor or
diverged — flagged in-plot). $x$ = mean accuracy across all 10 blocks.

In [ ]:
EPS_LO, EPS_HI = 0.005, 1.0   # a seed's own achievable scale must be in this range to trust a
                              # ratio: below EPS_LO it's indistinguishable from weight noise
                              # (e.g. OJA/BCM), above EPS_HI it's diverged (e.g. LC), not learned.

def trained_ref(d):
    """Per-seed |Δw| this rule reaches on whichever channel is its OWN active one that block,
    averaged over blocks 1..NR-1 -- the natural scale to judge everything else against, since
    PICKS gives each rule a very different alpha/beta scale."""
    eW = d['end_W']
    d0e = eW[:, :, 0, 1] - eW[:, :, 0, 0]; d1e = eW[:, :, 1, 1] - eW[:, :, 1, 0]
    own = np.where(np.arange(d0e.shape[1])[None, :] % 2 == 0, np.abs(d0e), np.abs(d1e))
    return own[:, 1:].mean(1)

def ratio_to_trained(vals_seed, ref_seed):
    out = np.full_like(vals_seed, np.nan, dtype=float)
    m = (ref_seed > EPS_LO) & (ref_seed < EPS_HI)
    out[m] = vals_seed[m] / ref_seed[m]
    return out, int(m.sum())

def held_opposite(d):
    eW = d['end_W']
    d0e = eW[:, :, 0, 1] - eW[:, :, 0, 0]; d1e = eW[:, :, 1, 1] - eW[:, :, 1, 0]
    heldA = -d0e[:, 1::2].mean(1)   # context A, measured during B-active (odd) blocks
    heldB = d1e[:, 2::2].mean(1)    # context B, measured during A-active (even) blocks, skip block 0
    return np.clip(np.minimum(heldA, heldB), -MCLIP, MCLIP)

Acc = {r: data[r]['acc'].mean(1) for r in ORDER}
Trained = {r: trained_ref(data[r]) for r in ORDER}
HeldOpp = {r: held_opposite(data[r]) for r in ORDER}
HeldOppR, HeldOppR_n = {}, {}
for r in ORDER:
    v, n = ratio_to_trained(HeldOpp[r], Trained[r])
    HeldOppR[r] = v; HeldOppR_n[r] = n

fig, ax = plt.subplots(figsize=(4.6, 3.2))
note = []
for r in ORDER:
    n = HeldOppR_n[r]
    if n == 0:
        note.append(f'{LABEL[r]}: no signal'); continue
    if n < SEEDS: note.append(f'{LABEL[r]}: n={n}/{SEEDS}')
    y = HeldOppR[r][np.isfinite(HeldOppR[r])]; x = Acc[r]
    ax.errorbar(x.mean(), y.mean(), xerr=x.std() / np.sqrt(len(x)), yerr=y.std() / np.sqrt(y.size),
                fmt='o', ms=6, color=CAT_COLOR[r], ecolor=CAT_COLOR[r], elinewidth=0.6, capsize=1.5, label=LABEL[r])
ax.axvline(0.5, color='#bbb', lw=0.5, ls='--'); ax.axhline(0, color='#bbb', lw=0.5, ls='--')
ax.margins(0.2)
nature_ax(ax, 'mean accuracy (all blocks)', 'held memory (opposite block) / trained', nx=4, ny=4)
if note:
    ax.text(0.02, 0.02, '\n'.join(note), transform=ax.transAxes, fontsize=6, color='#888', style='italic', va='bottom')
ax.legend(fontsize=7, frameon=False, loc='upper left', bbox_to_anchor=(1.02, 1.0))
if SAVE:
    fig.savefig(FIGDIR / 'F3_picks_opp_memory_vs_perf.svg', bbox_inches='tight')
plt.show()

In [ ]:
# coverage behind the "n=X/100" / "no signal" flags on both plots below: how many seeds had a
# usable (non-noise, non-diverged) own-scale to divide by. Same for both plots -- it depends only
# on Trained[r], not on which quantity is being normalized.
print(f"{'rule':4s} {'usable seeds':>14s}   (own trained-scale in ({EPS_LO}, {EPS_HI}))")
for r in ORDER:
    n = int(((Trained[r] > EPS_LO) & (Trained[r] < EPS_HI)).sum())
    print(f"{LABEL[r]:4s} {n:8d}/{SEEDS}")

## Fig. 3d: do ≥95% of simulations hold the inactive context?
Per-seed criterion instead of a test on the mean. A rule **holds** the inactive context if at least
95% of its usable seeds have an inactive-context preference **> 0** at the end of the last block —
equivalently, the 5th percentile across seeds is above 0, which is the lower whisker in the box
plot below. Usable seeds use the EPS_LO / EPS_HI filter defined above (trained scale in (0.005, 1.0)).

Alongside the descriptive pass/fail, an exact one-sided binomial test asks whether the *true*
fraction of seeds > 0 exceeds 95% (H₀: fraction ≤ 0.95), Holm-corrected across testable rules,
with a one-sided 95% Clopper–Pearson lower bound on that fraction. Note this is a demanding test:
with n = 100, only 100/100 (p ≈ 0.006) or 99/100 (p ≈ 0.04) seeds can reject H₀, and rules with
few usable seeds could not reach significance however clean they look.

Rules with fewer than `MIN_N` = 50 usable seeds (of 100) are **not tested**, the same as OJA:
with that few seeds there's no basis for a claim about 95% of simulations. This excludes BCM
(n = 4) and LC (n = 22); their rows are left empty in the box plot (as is OJA's).

In [ ]:
# ---- Fig. 3d (alternative): fraction of seeds with inactive-context preference > 0 ----
# Criterion: >= Q of usable seeds have inactive > 0 (== 5th percentile > 0 == lower whisker > 0).
# Exact one-sided binomial test of H0: true fraction <= Q, Holm-corrected.
# Rules with < MIN_N usable seeds are NOT tested (like OJA): too few seeds to say anything about 95%
# of simulations (e.g. BCM, n=4, would otherwise 'pass' on four near-zero seeds).
from scipy.stats import binomtest

def holm(pvals):
    """Holm-Bonferroni adjusted p-values (step-down, monotone, capped at 1)."""
    p = np.asarray(pvals, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); running = 0.0
    for k, i in enumerate(order):
        running = max(running, min(1.0, (m - k) * p[i]))
        adj[i] = running
    return adj

Q = 0.95
MIN_N = 50          # min usable seeds (of SEEDS) for a rule to be tested at all
WHIS = (100 * (1 - Q), 100 * Q)     # whiskers at 5th / 95th percentile -> criterion readable off the plot
last = NR - 1

def last_block_ratios(r):
    """Per-seed active (current) / inactive own-signed preference at end of the last block, as a fraction of
    that seed's trained scale (NaN where the seed's scale is noise-floor or diverged)."""
    eW = data[r]['end_W']
    mA = -(eW[:, last, 0, 1] - eW[:, last, 0, 0])
    mB = eW[:, last, 1, 1] - eW[:, last, 1, 0]
    cur_raw, inact_raw = (mB, mA) if last % 2 == 1 else (mA, mB)   # last block's active channel
    cur_r, _ = ratio_to_trained(cur_raw, Trained[r])
    inact_r, _ = ratio_to_trained(inact_raw, Trained[r])
    return cur_r[np.isfinite(cur_r)], inact_r[np.isfinite(inact_r)]

CUR, INACT, rows95 = {}, {}, []
for r in ORDER:
    CUR[r], INACT[r] = c, v = last_block_ratios(r)
    if v.size == 0:
        rows95.append(dict(rule=r, n=0)); continue
    k = int((v > 0).sum())
    bt = binomtest(k, v.size, Q, alternative='greater')
    rows95.append(dict(rule=r, n=v.size, k=k, frac=k / v.size, p5=np.percentile(v, WHIS[0]),
                       med=np.median(v), p=bt.pvalue,
                       lb=bt.proportion_ci(confidence_level=0.95, method='exact').low,
                       tested=v.size >= MIN_N))
    rows95[-1]['holds'] = rows95[-1]['tested'] and rows95[-1]['frac'] >= Q

testable95 = [i for i, row in enumerate(rows95) if row['n'] >= MIN_N]
for i, ph in zip(testable95, holm([rows95[i]['p'] for i in testable95])):
    rows95[i]['p_holm'] = ph

print(f"criterion: >= {Q:.0%} of usable seeds have inactive-context preference > 0; "
      f"rules with < {MIN_N} usable seeds not tested\n")
print(f"{'rule':5s} {'n':>4s} {'k>0':>4s} {'frac>0':>7s} {'95% LB':>7s} {'5th pct':>9s} {'median':>8s} "
      f"{'p (binom)':>10s} {'p (Holm)':>10s}  holds?")
for row in rows95:
    if row['n'] == 0:
        print(f"{LABEL[row['rule']]:5s} {0:4d}  not tested (no usable seeds)"); continue
    if not row['tested']:
        print(f"{LABEL[row['rule']]:5s} {row['n']:4d}  not tested (n < {MIN_N}; "
              f"{row['k']}/{row['n']} > 0, median {row['med']:.4f})"); continue
    print(f"{LABEL[row['rule']]:5s} {row['n']:4d} {row['k']:4d} {row['frac']:7.2f} {row['lb']:7.3f} "
          f"{row['p5']:9.4f} {row['med']:8.4f} {row['p']:10.2e} {row['p_holm']:10.2e}  "
          f"{'YES' if row['holds'] else 'no'}{' (binom sig.)' if row['p_holm'] < 0.05 else ''}")

# ---- box plot (horizontal), styled to match the edited Fig. 3d bar chart ----
# Rules stacked top-to-bottom; per rule the active env (green, top) above the inactive env (blue).
# Value axis on TOP, only left + top spines, Arial >= 7 pt, framed legend upper right.
# box = IQR, line = median, whiskers = EXACT 5th / 95th percentile (set by hand via bxp; plain
# ax.boxplot(whis=...) snaps whiskers to the nearest data point inside the bound, so the left
# whisker wouldn't be exactly the quantity the criterion tests), dots = seeds beyond the whiskers.
from matplotlib import cbook
XLIM = (-0.4, 3.0)   # fixed value range: LC's active-env seeds have diverged (5-95% ~ -2.2..7.7)
                     # and would squash every other rule; anything off-axis is printed below the table.
FS = 7               # minimum font size (pt) everywhere in this figure

def box_stats(v):
    st = cbook.boxplot_stats(v, whis=WHIS)[0]
    st['whislo'], st['whishi'] = np.percentile(v, WHIS)
    st['fliers'] = v[(v < st['whislo']) | (v > st['whishi'])]
    return st

ACTIVE_COL, INACT_COL = '#74ad83', '#7991b5'   # active env green, inactive env blue
# env names: A is trained on even blocks, B on odd -> block 10 (index NR-1 = 9) has B active
ACT_ENV, INACT_ENV = ('Env B', 'Env A') if last % 2 == 1 else ('Env A', 'Env B')
TESTED = {row['rule'] for row in rows95 if row.get('tested')}   # rows stay (labelled), boxes only for these
y = np.arange(len(ORDER))[::-1]                 # first rule in ORDER at the top
w = 0.3; GAP = 0.0                              # boxes of a pair touch (like the bars); wide gap between rules

with plt.rc_context({'font.family': 'Arial', 'font.size': FS,
                     'svg.fonttype': 'none'}):
    fig, ax = plt.subplots(figsize=(2.6, 3.2))
    offaxis, right_edge = [], {}
    for vals, dy, col, env in [(CUR, +w / 2 + GAP, ACTIVE_COL, ACT_ENV),
                               (INACT, -w / 2 - GAP, INACT_COL, INACT_ENV)]:
        rs = [r for r in ORDER if r in TESTED]   # untested rules (n < MIN_N, or none usable) are not drawn
        stats = [box_stats(vals[r]) for r in rs]
        bp = ax.bxp(stats, positions=[y[ORDER.index(r)] + dy for r in rs], widths=w, vert=False,
                    patch_artist=True, showfliers=True, manage_ticks=False,
                    boxprops=dict(facecolor=col, edgecolor='k', lw=0.5),
                    medianprops=dict(color='k', lw=0.8), whiskerprops=dict(color='k', lw=0.5),
                    capprops=dict(color='k', lw=0.5),
                    flierprops=dict(marker='o', ms=1.2, mfc=col, mec='none', alpha=0.8))
        bp['boxes'][0].set_label(env)
        for r, st in zip(rs, stats):
            if st['whislo'] < XLIM[0] or st['whishi'] > XLIM[1]:
                offaxis.append(f"{LABEL[r]} {env}: 5–95% = {st['whislo']:.1f}..{st['whishi']:.1f}")
            if vals is INACT:   # right end of what's drawn for the inactive box -> label goes after it
                vis = np.r_[st['whishi'], st['fliers']]
                right_edge[r] = min(vis[vis <= XLIM[1]].max(initial=st['whishi']), XLIM[1])

    ax.axvline(0, color='k', lw=0.5, zorder=0)
    for sp in ('bottom', 'right'): ax.spines[sp].set_visible(False)
    for sp in ('left', 'top'): ax.spines[sp].set_linewidth(0.5)
    ax.spines['left'].set_visible(False)        # the x = 0 line acts as the rule axis, as in the bar chart
    ax.xaxis.set_ticks_position('top'); ax.xaxis.set_label_position('top')
    ax.set_xticks([0, 1, 2, 3])
    ax.tick_params(axis='x', labelsize=FS, width=0.5, length=2)
    ax.set_yticks(y); ax.set_yticklabels([LABEL[r] for r in ORDER], fontsize=FS)
    ax.tick_params(axis='y', length=0, pad=2)
    ax.set_xlabel('relative weight / trained', fontsize=FS)
    ax.set_xlim(*XLIM); ax.set_ylim(-0.6, len(ORDER) - 0.4)

    # % of seeds > 0 just right of each inactive box -- only for rules that meet the >=95% criterion
    # (failing / untested / no-signal rules get no label; see the printed table for their numbers)
    for i, row in enumerate(rows95):
        if not row.get('holds'):
            continue
        r = row['rule']
        txt = f"{row['frac']:.0%}" + ('' if row['n'] == SEEDS else f" (n={row['n']})")
        ax.text(right_edge[r] + 0.06, y[i] - w / 2 - GAP, txt, ha='left', va='center', fontsize=FS,
                clip_on=False, color='k', fontweight='bold')

    h, l = ax.get_legend_handles_labels()
    order = [l.index('Env A'), l.index('Env B')]            # Env A first, as in the bar chart
    leg = ax.legend([h[k] for k in order], [l[k] for k in order], fontsize=FS, loc='upper right',
                    frameon=True, fancybox=False, edgecolor='k', handlelength=1.0, handleheight=0.8,
                    borderpad=0.4)
    leg.get_frame().set_linewidth(0.5)
    if offaxis:
        print('off-axis (whiskers beyond XLIM):', '; '.join(offaxis))
    if SAVE:
        fig.savefig(FIGDIR / 'F3_picks_context_weights_box.svg', bbox_inches='tight')
    plt.show()